# Monday: describing patterns in text

How can we describe patterns across more texts than we can read individually?

Work with a partner if you can. Start with three short texts, then compare AITA replies using a word search, counts, and TF-IDF. Write code in the empty cells and use the supplied examples as references. Run those examples yourselves; the checks tell you what to expect.

Complete Parts 1–5 first. Extra experiments are at the end. If you work in pairs, switch who types after Part 3. Make a copy named `week08_live_workspace_mine.ipynb`.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

## 1. From texts to features

A **document** is one text we want to describe. A **feature** is something we measure consistently for each document. Here, our features will be word counts.

The same three documents are in the Week 8 slides, linked from the Week 8 README. Before running code: how many times does **the** appear in each document? Which two documents share more words?

In [ ]:
sentences = [
    "The cat sat on the mat.",
    "The cat sat on the rug.",
    "The capital of France is Paris."
]

### Make three features yourself

Use `pd.DataFrame()` to make a table called `manual_counts` with three columns: `cat`, `mat`, and `rug`. Each column should contain the counts for documents 1, 2, and 3, in that order. Show the table.

For example, a dictionary entry looks like `"cat": [1, 1, 0]`. What does a zero mean?

In [ ]:
# Make and show manual_counts.

### Build the full count table

`CountVectorizer` builds a **vocabulary** of words and counts each word in each document. These defaults lowercase words and ignore punctuation as word features.

Read the supplied code, then run it yourselves:

1. `CountVectorizer()` creates the counter.
2. `fit_transform(sentences)` learns the word columns and counts occurrences in each text.
3. `toarray()` gives the numbers to pandas; `get_feature_names_out()` gives the column names.

The result is a **document–word matrix**. Rows are documents, columns are words, and cells are counts. Check: 3 rows and 11 word columns. Find your three manual columns in this larger table.

In [ ]:
counter = CountVectorizer()
count_matrix = counter.fit_transform(sentences)
count_table = pd.DataFrame(count_matrix.toarray(), columns=counter.get_feature_names_out())
count_table

## 2. Use the numbers

Write code to answer these questions using `count_table`:

1. How many documents and word columns are there? Use `.shape`.
2. How many times does **the** occur across all three documents? Select its column and use `.sum()`.
3. In how many documents does **cat** appear? Make `present = count_table > 0`, then find the sum and mean of its `cat` column.

What is the difference between counting occurrences and counting documents?

In [ ]:
# Show the shape, total the count, and cat presence sum and mean.

### Experiment: repetition and word order

Run the next cell to make two versions of document 1. In the first, **cat** is repeated. In the second, the original words are rearranged.

Predict the counts of `cat`, `mat`, and `the` in each version. Then write code to:

1. Make `changed_matrix` using `counter.transform(changed)`. This counts using the existing word columns.
2. Make a DataFrame called `changed_counts` using the same construction as above.
3. Show just its `cat`, `mat`, and `the` columns.

Check: the `cat` counts are 2 and 1; `mat` is 1 in both; `the` is 2 in both. Which change affects the counts?

In [ ]:
changed = [
    "The cat cat sat on the mat.",
    "On the mat sat the cat."
]

In [ ]:
# Count changed, build changed_counts, and show cat, mat and the.

### Choose what to record

The **vocabulary** determines the word columns. A **stop-word list** removes chosen words from those columns. **Bigrams** record adjacent pairs of words.

If we remove `the`, which columns disappear? If we keep only `cat`, `mat`, and `rug`, what information do we retain from document 3?

## 3. From counts to weights

**The** appears in every document. **Mat** appears in only one. Words found in fewer documents can help distinguish them.

**TF-IDF** combines term frequency (the count within a document) with inverse document frequency (a larger factor for words found in fewer documents). The default also rescales each numeric row to a common length. This changes the numbers; it does not change how many words a text contains. The weights are not percentages and need not sum to one.

Adapt the count example: use `TfidfVectorizer()` to create `toy_tfidf`, use its `fit_transform(sentences)` to create `toy_matrix`, and build a DataFrame called `toy_weights`. Show its `the`, `cat`, and `mat` columns with `.round(2)`.

Check: document 1 has cat = 0.37 and mat = 0.49. Which cells stay zero? Cat and mat both have count 1 here: why does mat get the higher weight?

In [ ]:
# Build toy_weights. Show its the, cat and mat columns.

## 4. Apply the idea: compare AITA replies

Do replies beginning with **NTA** and **YTA** mention wife at different rates? We will create a word-presence feature and summarize it using `groupby`.

One row is one reply to an AITA post. This file contains 1,512 screened, highly scored historical direct replies. We will use the 1,035 with a leading NTA or YTA marker. NTA means “not the asshole”; YTA means “you’re the asshole.” These markers provide our grouping information.

Inspect the texts, write a search, compare group shares, then read a full reply from each group. Write your code in the empty cells. The checks show what to expect.

In [ ]:
url = "https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/docs/interactives/data/week08/aita_replies.csv"
comments = pd.read_csv(url)
pd.set_option("display.max_colwidth", 300)  # show more of each reply
comments.head(3)

In [ ]:
# Count the replies with missing text in body. Check: 0.

### Make the two groups

The next cell finds replies that **start with** NTA or YTA. `^` means the start of the text, `|` means either choice, and `\b` marks the end of a word. Replies without either marker get a missing value.

In [ ]:
comments["marker"] = comments["body"].str.upper().str.extract(r"^(NTA|YTA)\b")
comments["marker"].value_counts(dropna=False)

In [ ]:
# This reply mentions NTA, but not at the start.
comments.loc[comments["idstr"] == "t1_ed0g14e", ["body", "marker"]]

Without `^`, which group would this reply get? Why might a verdict found anywhere in a reply put it in the wrong group?

Keep the replies with a marker in a new table called `replies`. Use `dropna(subset=[...]).copy()`.

In [ ]:
# Make replies.

In [ ]:
replies["marker"].value_counts()  # Check: NTA 657, YTA 378

### Search for a word

`.str.contains()` asks whether each text matches a search. The next cell adds two searches as columns to a small table of example sentences. Why does the first search match **midwife**? What does `\b` change?

In [ ]:
examples = pd.DataFrame({"text": ["His wife agrees.", "The midwife agrees.", "His partner agrees."]})
examples["wife"] = examples["text"].str.contains("wife", case=False)
examples["whole_word"] = examples["text"].str.contains(r"\bwife\b", case=False)  # r"..." keeps the backslash
examples

Now search the replies. Make a column `replies["mentions"]` that is True for each reply containing the whole word **wife**.

In [ ]:
# Make replies["mentions"] with the pattern r"\bwife\b".

In [ ]:
print(replies["mentions"].sum())  # Check: 56

`True` counts as 1 and `False` as 0, so the mean of `mentions` is the share of replies that match. Use `groupby` to show, for each marker, the number of replies, the number of matches, and the share.

*Hint: `.agg(n_replies="size", n_matches="sum", share="mean")`*

In [ ]:
# Count replies, matches and the share in each group.
# Check: NTA 657 replies, 27 matches. YTA 378 replies, 29 matches.

The two groups have almost the same number of matches. Which group mentions wife more often? Why do the counts and the shares give different answers?

### Read a full reply from each group

A share does not show how a word is used. The next cell selects YTA matches, displays their IDs, and prints the first reply in full. Tables can cut off long text, so use `print()` to read the complete reply.

In [ ]:
yta_matches = replies[replies["mentions"] & (replies["marker"] == "YTA")]
yta_matches[["idstr", "body"]].head()
print(yta_matches.iloc[0]["idstr"])
print(yta_matches.iloc[0]["body"])

In [ ]:
# Adapt the cell above for NTA. Print its first matching ID and full reply.

Who is the wife in these replies, and how is she described in each group? Write down one ID from each group that shows the difference.

### Your finding

Write two sentences reporting the group shares and describing how wife is discussed in the two full replies. Include their IDs. Keep the claim about this sample.

Your finding:

## 5. Compare every word: counts and TF-IDF

So far we chose one word. Now use a vectorizer to count the vocabulary across all 1,035 replies.

Run the next cell yourselves. It follows the small count example. `stop_words="english"` removes chosen common words, and `min_df=5` keeps words found in at least five replies. `index=replies.index` keeps the document rows aligned with the reply table.

Check: 1,035 rows and 773 word columns.

In [ ]:
reply_counter = CountVectorizer(stop_words="english", min_df=5)
reply_counts = reply_counter.fit_transform(replies["body"])
count_words = pd.DataFrame(reply_counts.toarray(), columns=reply_counter.get_feature_names_out(), index=replies.index)
count_words.shape

### Summarize word columns by group

Previously, you took the mean of one True/False column. This time, take the mean of **all the word-count columns**.

Run the next cell, then inspect three columns. The result has two rows, NTA and YTA; the columns are still words. Each value is the average number of occurrences per reply in that group. Repeating a word twice adds two occurrences.

Which group has the higher mean count for wife?

In [ ]:
count_means = count_words.groupby(replies["marker"]).mean()
count_means[["wife", "friend", "yta"]].round(3)

### Rank differences between the groups

`count_means.loc["YTA"]` selects the YTA row: one mean for each word. Subtracting the NTA row compares the same word columns. Positive means a higher mean count in YTA replies; negative means a higher mean in NTA replies.

Run the supplied cell to sort the differences from largest to smallest.

In [ ]:
count_difference = count_means.loc["YTA"] - count_means.loc["NTA"]
count_difference.sort_values(ascending=False).head(10)

Why is **yta** at the top? Where would you expect **nta**?

### Your code: compare TF-IDF weights

Copy the three short count cells above into the next cell. Change `CountVectorizer` to `TfidfVectorizer`, keeping `stop_words="english"` and `min_df=5`. Use these names consistently:

| Counts name | TF-IDF name |
|---|---|
| `reply_counter` | `reply_tfidf` |
| `reply_counts` | `reply_weights` |
| `count_words` | `tfidf_words` |
| `count_means` | `tfidf_means` |
| `count_difference` | `tfidf_difference` |

Build the weights table, average within groups, and rank YTA minus NTA. Keep the count results so that you can compare the two lists.

In [ ]:
# Compare the groups with TfidfVectorizer. Check: the top 10 starts yta, way, time, wife.

In [ ]:
lists = pd.DataFrame({
    "counts": count_difference.sort_values(ascending=False).head(15).index,
    "tfidf": tfidf_difference.sort_values(ascending=False).head(15).index,
})
lists

Name two words shared by the lists. Find one word that changes rank or appears in only one list. Where is **wife**?

Then calculate the mean number of words per reply in each group. Use `replies["body"].str.split().str.len()` to get one word count per reply, and group those counts by `replies["marker"]`.

In [ ]:
# Mean number of words per reply in each group. Hint: replies["body"].str.split().str.len()
# Check: NTA 45.2, YTA 54.1

YTA replies average 54.1 words; NTA replies average 45.2. Longer replies offer more opportunities for a word to occur. How could this affect the count comparison?

TF-IDF gives a smaller IDF factor to words found in many replies and rescales the numeric rows. Pick one term whose ranking changed. Which list would you use to choose a source reply to read, and why?

### Before Friday

On Friday you choose a question about word use across two groups, build a word-presence feature, compare group shares, read examples, and change one search rule.

You can use your project texts. If you want to, one person per team makes a file called `lab08_team_texts.csv` with three columns, `id`, `text`, and `group`, and about 100–300 rows. Leave out texts with names, usernames, or other details that identify a person. Save it in the `lab` folder, beside the Lab 8 notebook. Teams without a file use the AITA replies.

If your project data is already a CSV, this code makes the file. Change the file and column names to yours:

```python
texts = pd.read_csv("your_project_data.csv")
texts = texts.rename(columns={"your_id_column": "id", "your_text_column": "text", "your_group_column": "group"})
texts = texts.dropna(subset=["text", "group"])
texts[["id", "text", "group"]].sample(300, random_state=1).to_csv("lab08_team_texts.csv", index=False)
```

With fewer than 300 rows, change 300 to the number of rows you have.

You have completed the core notebook. The experiments below are optional.

## Extra experiments

Try these if you have time.

**Optional experiment:** add a fourth document, `The cat sat on the mat.`, and fit a new TF-IDF vectorizer on all four. Has the weight for `mat` in document 1 changed?

In [ ]:
# Optional: add the document, refit, and compare.

### If you have time: change the rule

Write a pattern that matches **wife or partner** as whole words. Add it as a column to `examples` first, and predict which of the three sentences it will match.

`|` means either choice. Put `\b` around each word: `\bword1\b|\bword2\b`.

In [ ]:
# Add a column to examples with your new pattern. Check: True, False, True.

In [ ]:
# Make replies["mentions_v2"] with your new pattern. Check: 69 replies match.

In [ ]:
replies.groupby("marker")[["mentions", "mentions_v2"]].mean()

How did the shares change? Did one group change more than the other?

Now show the replies that only the new rule matches: `mentions_v2` is True and `mentions` is False. *Hint: `&` combines conditions, and `~` reverses a True/False condition. Put each condition in parentheses.*

In [ ]:
# Show idstr, marker and body for the replies only mentions_v2 matches. Check: 13 replies.

Read three of them. Is **partner** used for the same kind of person as wife? Did widening the rule help, or did it add a different meaning?

If you have time: read a few replies that neither rule matches. Does any of them discuss the same idea in other words?

In [ ]:
# Optional: read replies that neither rule matches.